In [ ]:
# PAWN SDXL warm-session kernel — load SDXL ONCE, then serve a work-loop
# against the self-hosted PostgREST instance (fast repeat images while the
# container stays alive). PAWN replaces __PAWN_PAYLOAD_B64__ with
# base64(json) before the push.
import base64, json, time, datetime, io
import requests  # preinstalled on Kaggle

payload = json.loads(base64.b64decode("__PAWN_PAYLOAD_B64__").decode())

SESSION_ID = payload["session_id"]
POSTGREST_URL = payload["postgrest_url"].rstrip("/")
POLL = int(payload.get("poll_interval", 3))

# Anonymous requests to PostgREST get the restricted `pawn_anon` Postgres role
# (RLS permits it on the two dedicated tables only) — no bearer/API key needed.
REST = POSTGREST_URL
HEADERS = {
    "Content-Type": "application/json",
}

def now_iso():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def now_utc():
    return datetime.datetime.now(datetime.timezone.utc)

def parse_ts(s):
    if not s:
        return None
    try:
        return datetime.datetime.fromisoformat(s.replace("Z", "+00:00"))
    except ValueError:
        return None

def get_session():
    r = requests.get(f"{REST}/image_sessions", headers=HEADERS,
                     params={"id": f"eq.{SESSION_ID}", "select": "*"}, timeout=20)
    r.raise_for_status()
    data = r.json()
    return data[0] if data else None

def patch_session(fields):
    requests.patch(f"{REST}/image_sessions", headers=HEADERS,
                   params={"id": f"eq.{SESSION_ID}"}, json=fields, timeout=20)

def next_job():
    r = requests.get(f"{REST}/image_jobs", headers=HEADERS,
                     params={"session_id": f"eq.{SESSION_ID}", "status": "eq.queued",
                             "order": "created_at.asc", "limit": "1", "select": "*"},
                     timeout=20)
    r.raise_for_status()
    data = r.json()
    return data[0] if data else None

def patch_job(job_id, fields):
    requests.patch(f"{REST}/image_jobs", headers=HEADERS,
                   params={"id": f"eq.{job_id}"}, json=fields, timeout=30)

def png_b64(image):
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

import threading

def heartbeat_during(interval=20):
    """Return a context manager that sends a session heartbeat every `interval`
    seconds in a background thread. Use it to keep the session alive while a
    long blocking call (e.g. pipe()) runs.

    Usage:
        with heartbeat_during():
            image = pipe(...).images[0]
    """
    stop_evt = threading.Event()

    def _run():
        while not stop_evt.wait(interval):
            try:
                patch_session({"heartbeat_at": now_iso()})
            except Exception:
                pass  # network blip — next tick will retry

    class _Ctx:
        def __enter__(self):
            self._t = threading.Thread(target=_run, daemon=True)
            self._t.start()
            return self
        def __exit__(self, *_):
            stop_evt.set()
            self._t.join(timeout=5)

    return _Ctx()


In [ ]:
patch_session({"status": "installing"})
import subprocess, sys
print("Installing SDXL dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "diffusers", "transformers", "accelerate"])
print("Dependencies installed.")


In [ ]:
# Load SDXL ONCE (a single T4 fits the ~7GB fp16 weights). On success announce
# 'ready'; on failure record the error on the session row and exit (no hang).
patch_session({"status": "loading_model"})
import os, torch
try:
    from diffusers import AutoPipelineForText2Image

    input_base = "/kaggle/input"
    target_dir = None
    for root, dirs, files in os.walk(input_base):
        if "model_index.json" in files:
            target_dir = root
            break
    if not target_dir:
        raise FileNotFoundError("SDXL dataset not mounted (no model_index.json found).")

    print(f"Loading SDXL pipeline from {target_dir}...")
    pipe = AutoPipelineForText2Image.from_pretrained(
        target_dir,
        torch_dtype=torch.float16,
        use_safetensors=True,
        local_files_only=True,
    )
    pipe = pipe.to("cuda")
    pipe.set_progress_bar_config(disable=True)
    print("SDXL loaded; session is ready.")
    patch_session({"status": "ready", "heartbeat_at": now_iso()})
except Exception as e:
    patch_session({"status": "error", "error": f"model load failed: {e}"})
    raise


In [ ]:
# Serve loop: heartbeat, honor stop/timer/cap, generate each pending job FAST
# (model already warm) and write the PNG back into its row.
images_done = 0
while True:
    sess = get_session()
    if sess is None:
        print("session row gone; exiting")
        break
    status = sess.get("status")
    expires = parse_ts(sess.get("expires_at"))
    cap = sess.get("max_images")
    if status in ("stopping", "ended"):
        patch_session({"status": "ended"})
        print("stop requested; exiting")
        break
    if expires is not None and now_utc() >= expires:
        patch_session({"status": "ended"})
        print("timer expired; exiting")
        break
    if cap is not None and images_done >= cap:
        patch_session({"status": "ended"})
        print("image cap reached; exiting")
        break
    patch_session({"heartbeat_at": now_iso()})
    job = next_job()
    if not job:
        time.sleep(POLL)
        continue
    job_id = job["id"]
    patch_job(job_id, {"status": "running", "started_at": now_iso()})
    try:
        p = job.get("params") or {}
        init_b64 = p.get("init_image_b64")
        with heartbeat_during(interval=20):
            if init_b64:
                from diffusers import AutoPipelineForImage2Image
                from PIL import Image
                init_image = Image.open(io.BytesIO(base64.b64decode(init_b64))).convert("RGB")
                img2img_pipe = AutoPipelineForImage2Image.from_pipe(pipe)
                image = img2img_pipe(
                    prompt=job["prompt"],
                    image=init_image,
                    strength=p.get("strength", 0.6),
                    negative_prompt=p.get("negative_prompt", ""),
                    num_inference_steps=p.get("num_inference_steps", 30),
                    guidance_scale=p.get("guidance_scale", 7.5),
                ).images[0]
            else:
                image = pipe(
                    prompt=job["prompt"],
                    negative_prompt=p.get("negative_prompt", ""),
                    num_inference_steps=p.get("num_inference_steps", 30),
                    guidance_scale=p.get("guidance_scale", 7.5),
                    width=p.get("width", 1024),
                    height=p.get("height", 1024),
                ).images[0]
        patch_job(job_id, {"status": "done", "image_b64": png_b64(image),
                           "mime": "image/png", "via": "kaggle:sdxl-session",
                           "done_at": now_iso()})
        images_done += 1
        patch_session({"images_done": images_done})
    except Exception as e:
        patch_job(job_id, {"status": "error", "error": str(e), "done_at": now_iso()})